# 🎯 [C202_M08] 대역폭 90% 급감 및 대량 전문 폭주 상황 모의 전송 스트레스 시뮬레이션
### Google Colab 실습 환경 (지휘통제(C2) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C202/c202_m08_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 2) |
| **미션 ID** | C202_M08 |
| **미션 주제** | 대역폭 90% 급감 및 대량 전문 폭주 상황 모의 전송 스트레스 시뮬레이션 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 전자전 재밍과 통신 장비 부족으로 협대역 무선망의 대역폭이 크게 줄어드는 상황이 잦았고, 이때 적 접촉 보고나 의무후송 요청 같은 긴급 전문이 행정 전문 뒤에 묶여 늦게 전달되는 문제가 지휘통제의 약점이 되었다. 본 프로젝트에서는 가상 협대역 링크(2.4kbps)와 가상 전술 전문을 코드로 직접 만들어, 전문 구조 정의, 패킷 파싱·압축 전처리, 긴급도 기반 우선순위 큐, 채널 상태에 따른 지능형 재전송, 링크 신뢰도 산출, 통신 관제 화면, 통신관 HITL 통제, 대역폭 급감 스트레스 시험, AAR 재조정, 최종 통합까지 메시지 우선순위 큐·재전송 프로토타입을 단계별로 구현한다. 최종 목표는 대역폭이 제한된 상황에서도 FLASH·IMMEDIATE 전문을 기한 안에 전달하고, 어떤 전문도 잃어버리지 않으며, 순서 조정의 최종 판단은 통신관이 내리도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c202_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c202.py` | 전처리, 우선순위 큐, 재전송 제어, 링크 신뢰도, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c202_data_spec.md`, `docs/c202_hitl_guide.md`, `docs/c202_aar_report.md`, `docs/c202_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c202_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c202_stress_test.py` | 대역폭 90% 급감 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c202_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 08. 대역폭 90% 급감 및 대량 전문 폭주 상황 모의 전송 스트레스 시뮬레이션
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

대역폭 90% 급감 및 대량 전문 동시 발생 극한 스트레스 시뮬레이션을 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **대역폭 90% 급감과 전문 트래픽 500% 증가, 재밍·링크 두절 상황에서 큐·재전송 시스템의 안전 실패 동작을 검증한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **평시, 트래픽 6배, 대역폭 10%로 급감, 두 조건 동시, 재밍 120초, 대역폭 1% 두절 상황을 가상으로 주입하고 일반 모드와 성능 저하 모드(긴급 전문만 송신, 작은 조각·압축 헤더, 나머지 보관)를 비교 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `tests/c202_stress_test.py 및 스트레스 테스트 분석 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 극단 상황에서의 안전 실패(Fail-safe) 및 기능 축소 작동 적절성

#### 미션 PASS 조건

- 정량 기준: 동시 트래픽 500% 증가 시 시스템 다운 0건, 데이터 유실률 0%


- 진행 기준: 대량 시뮬레이션 부하 주입 테스트 통과 및 병목 구간 프로파일링 완료


- 검증 방법: 부하 테스트 도구를 통한 처리량(Throughput) 및 메모리 누수 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""대역폭 90% 급감, 전문 트래픽 500% 증가(평시 6배), 재밍·링크 두절을 가상으로 주입해 큐·재전송 시스템이 멈추지 않는지(다운 0건), 전문이 사라지지 않는지(유실 0건: 전달+보관=전체), 성능 저하 모드(긴급 전문만 송신·작은 조각·압축 헤더, 나머지 보관)가 긴급 전문 전달을 지키는지 검증하고 처리량·메모리·병목 함수를 측정한다."""
import cProfile
import io
import pstats
import time
import tracemalloc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 전술 전문(메시지) 생성기 (실제 부대·통신망·제원 정보 없음)
# 긴급도 4단계(FLASH/IMMEDIATE/PRIORITY/ROUTINE)와 전문 유형별 크기를 가진 전문을 만든다.
# ---------------------------------------------------------------------------
PREC = ["FLASH", "IMMEDIATE", "PRIORITY", "ROUTINE"]
RANK = {p: i for i, p in enumerate(PREC)}
DEADLINE = {"FLASH": 10, "IMMEDIATE": 30, "PRIORITY": 120, "ROUTINE": 600}   # 가상 전달 목표 시간(초)
MSG_TYPES = {   # 유형: (긴급도 후보, 확률, 본문 크기 범위 bytes, 발생 비율)
    "CONTACT": (["FLASH", "IMMEDIATE"], [0.7, 0.3], (120, 300), 0.08),
    "MEDEVAC": (["FLASH", "IMMEDIATE"], [0.4, 0.6], (150, 350), 0.05),
    "SITREP": (["IMMEDIATE", "PRIORITY"], [0.2, 0.8], (600, 2500), 0.20),
    "POSREP": (["PRIORITY", "ROUTINE"], [0.5, 0.5], (60, 120), 0.35),
    "LOGREQ": (["PRIORITY", "ROUTINE"], [0.4, 0.6], (300, 1200), 0.14),
    "ADMIN": (["ROUTINE"], [1.0], (1000, 5000), 0.18),
}


def make_messages(duration=600, rate=0.12, seed=7, t_start=0.0, prefix="M"):
    """duration초 동안 평균 rate건/초로 발생하는 가상 전문 목록."""
    rng = np.random.default_rng(seed)
    n = int(rng.poisson(rate * duration))
    times = np.sort(rng.uniform(t_start, t_start + duration, n))
    types = list(MSG_TYPES)
    msgs = []
    for i, t0 in enumerate(times):
        ty = types[rng.choice(len(types), p=[MSG_TYPES[k][3] for k in types])]
        cands, p, (lo, hi), _ = MSG_TYPES[ty]
        msgs.append({"id": f"{prefix}{i:04d}", "t0": float(t0), "type": ty, "prec": str(rng.choice(cands, p=p)),
                     "size": int(rng.integers(lo, hi + 1)), "src": f"U{int(rng.integers(1, 9)):02d}"})
    return msgs


class Link:
    """[교육용가상자료] 가상 협대역 무선 링크(기본 2.4kbps). 초 단위 상태(good/bad/jam)에 따라
    비트오류율이 바뀌고(Gilbert-Elliott 모델), 구간별 대역폭 감소도 줄 수 있다."""
    BER = {"good": 1e-5, "bad": 2e-4, "jam": 1.2e-3}

    def __init__(self, horizon=4000, bw_bps=2400, seed=1, p_gb=0.02, p_bg=0.15, jam=(), bw_drop=(), rtt=(1.2, 0.4)):
        rng = np.random.default_rng(seed)
        st, s = [], "good"
        for _ in range(int(horizon) + 1):
            s = ("bad" if rng.random() < p_gb else "good") if s == "good" else ("good" if rng.random() < p_bg else "bad")
            st.append(s)
        for a, b in jam:
            for k in range(int(a), min(int(b), len(st))):
                st[k] = "jam"
        self.states, self.bw_bps, self.bw_drop, self.rtt_par = st, bw_bps, list(bw_drop), rtt

    def state(self, t):
        return self.states[min(int(t), len(self.states) - 1)]

    def bw(self, t):
        f = 1.0
        for a, b, fac in self.bw_drop:
            if a <= t < b:
                f = fac
        return self.bw_bps * f

    def p_loss(self, nbytes, t):
        return 1 - (1 - self.BER[self.state(t)]) ** (8 * nbytes)

    def rtt(self, rng):
        return self.rtt_par[0] + rng.exponential(self.rtt_par[1])
# ---------------------------------------------------------------------------

# ---- 우선순위 큐·재전송 이산 사건 시뮬레이터 공통 함수 ----
HDR = 32   # 전문 조각(패킷) 헤더 bytes (미션 2의 헤더 압축 적용 시 12 bytes)


def policy_key(m, t, policy, aging):
    # [TODO : 학생 실습 8] 대역폭 90% 급감 및 대량 전문 폭주 상황 모의 전송 스트레스 시뮬레이션 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def simulate(msgs, link, policy="strict", mtu=256, preempt=True, rto_mode="fixed", rto_s=4.0, aging=0.0,
             redundancy=False, adapt_mtu=False, hdr=HDR, seed=0, until=None, admit=None):
    """전문 목록을 링크로 보내는 이산 사건 시뮬레이션.
    - 전문은 mtu 크기 조각으로 나눠 보낸다. 조각이 손실되면 RTO 후 재전송한다(자료는 버리지 않음).
    - preempt=False면 한 전문의 조각을 끝까지 연속 전송한다(큰 전문이 긴급 전문을 막는 현상 재현).
    - rto_mode='adaptive'면 왕복시간(RTT) 추정값으로 RTO를 계산한다(Jacobson 방식).
    - redundancy=True면 링크 열화 추정 시 FLASH·IMMEDIATE 조각을 2회 연속 전송한다.
    - adapt_mtu=True면 링크 열화 추정 시 조각 크기를 줄인다.
    - admit(m, t)이 False인 전문은 새 조각을 보내지 않고 저장해 둔다(성능 저하 모드)."""
    rng = np.random.default_rng(seed)
    pend = sorted(({**m, "unsent": m["size"], "ready": [], "frags": [], "done": None, "first": None} for m in msgs), key=lambda m: m["t0"])
    t, i, active, cur = 0.0, 0, [], None
    srtt, rttvar, loss = None, None, 0.0
    st = {"tx": 0, "retx": 0, "spurious": 0, "bytes": 0, "decide_ms": [], "flag": {}, "log": []}
    until = until if until is not None else (pend[-1]["t0"] if pend else 0) + 6000
    while (i < len(pend) or active) and t < until:
        while i < len(pend) and pend[i]["t0"] <= t:
            active.append(pend[i]); i += 1
        td = time.perf_counter()
        for m in active:                                   # ACK가 이미 도착한 재전송 예약은 취소
            m["ready"] = [e for e in m["ready"] if not (e[1]["ack"] is not None and e[1]["ack"] <= t)]
        ready = [m for m in active if any(r <= t for r, _ in m["ready"]) or (m["unsent"] > 0 and (admit is None or admit(m, t)))]
        if not ready:
            nxt = ([pend[i]["t0"]] if i < len(pend) else []) + [r for m in active for r, _ in m["ready"] if r > t]
            t = min(nxt) if nxt else t + 1.0
            continue
        if not preempt and cur in ready and cur["unsent"] > 0:
            m = cur
        else:
            m = min(ready, key=lambda x: policy_key(x, t, policy, aging))
        st["decide_ms"].append((time.perf_counter() - td) * 1000)
        due = [e for e in m["ready"] if e[0] <= t]
        if due:
            e = min(due, key=lambda x: x[0]); m["ready"].remove(e); frag = e[1]
            st["retx"] += 1; st["spurious"] += frag["recv"]
        else:
            size = mtu if not adapt_mtu else (mtu if loss < 0.15 else (mtu // 2 if loss < 0.4 else mtu // 4))
            size = min(m["unsent"], size); m["unsent"] -= size
            frag = {"b": size, "recv": False, "ack": None, "tries": 0}
            m["frags"].append(frag)
        cur = m
        m["first"] = t if m["first"] is None else m["first"]
        nb = frag["b"] + hdr
        copies = 2 if (redundancy and RANK[m["prec"]] <= 1 and loss > 0.15) else 1
        for _ in range(copies):
            t += nb * 8 / link.bw(t)
            st["tx"] += 1; st["bytes"] += nb
            ok = rng.random() >= link.p_loss(nb, t)
            st["log"].append((t, nb, ok, m["prec"], m["id"]))            # 전송 기록(시각, bytes, 성공, 긴급도, 전문)
            loss = 0.85 * loss + 0.15 * (0.0 if ok else 1.0)          # 최근 손실률 추정(EWMA)
            st["flag"][int(t)] = st["flag"].get(int(t), False) or loss > 0.12   # 링크 열화 판정(초 단위)
            if ok and not frag["recv"]:
                frag["recv"], frag["rt"] = True, t
                r = link.rtt(rng); frag["ack"] = t + r
                if srtt is None:
                    srtt, rttvar = r, r / 2
                else:
                    rttvar = 0.75 * rttvar + 0.25 * abs(srtt - r); srtt = 0.875 * srtt + 0.125 * r
        frag["tries"] += 1
        rto = rto_s if (rto_mode == "fixed" or srtt is None) else float(np.clip(srtt + 4 * rttvar, 1.0, 30.0))
        if not frag["recv"] or frag["ack"] > t + rto:                   # 손실 또는 ACK 지연 -> 재전송 예약
            m["ready"].append((t + rto, frag))
        if m["done"] is None and m["unsent"] == 0 and all(f["recv"] for f in m["frags"]):
            m["done"] = max(f["rt"] for f in m["frags"])
        active = [x for x in active if not (x["unsent"] == 0 and not x["ready"] and all(f["recv"] for f in x["frags"]))]
    res = [{"id": m["id"], "type": m["type"], "prec": m["prec"], "size": m["size"], "t0": m["t0"], "done": m["done"], "wait": None if m["first"] is None else m["first"] - m["t0"],
            "retx": sum(f["tries"] for f in m["frags"]) - len(m["frags"]),
            "latency": None if m["done"] is None else m["done"] - m["t0"]} for m in pend]
    for r in res:
        r["on_time"] = r["latency"] is not None and r["latency"] <= DEADLINE[r["prec"]]
    return res, st


def summarize(res):
    """긴급도별 기한 내 전달률·지연. 종합 정확도 = 긴급도별 기한 내 전달률의 평균."""
    # [TODO : 학생 실습 8] 대역폭 90% 급감 및 대량 전문 폭주 상황 모의 전송 스트레스 시뮬레이션 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
ADV = dict(policy="aging", aging=0.5, rto_mode="adaptive", redundancy=True, adapt_mtu=True)
NOMINAL_BPS = 2400


def degraded_admit(link):
    """성능 저하 모드: 대역폭이 평시의 50% 미만이거나 링크가 재밍 상태면 FLASH·IMMEDIATE만 새로 송신하고 나머지는 보관(삭제 없음)."""
    return lambda m, t: RANK[m["prec"]] <= 1 or (link.bw(t) >= 0.5 * NOMINAL_BPS and link.state(t) != "jam")


def run_case(msgs, link, fail_safe, seed=0):
    t0 = time.perf_counter()
    try:
        kw = dict(mtu=96, hdr=14, admit=degraded_admit(link)) if fail_safe else {}    # 저하모드: 작은 조각 + 압축 헤더(미션 2) + 긴급 전문만 송신
        res, st = simulate(msgs, link, **ADV, seed=seed, until=2000, **kw)
        down = 0
    except Exception as e:                                       # 다운 집계
        print("다운:", e); res, st, down = [], {"tx": 0}, 1
    dt = time.perf_counter() - t0
    delivered = sum(r["done"] is not None for r in res)
    kept = len(res) - delivered                                   # 2,000초 시점까지 미전달 = 저장·보관 중(삭제 없음)
    urg = [r["on_time"] for r in res if RANK[r["prec"]] <= 1]
    return {"전문": len(msgs), "전달": delivered, "보관": kept, "유실": len(msgs) - delivered - kept, "다운": down,
            "긴급 기한내": round(float(np.mean(urg)), 3) if urg else np.nan,
            "FLASH 95%지연(s)": round(float(np.quantile([r["latency"] for r in res if r["prec"] == "FLASH" and r["latency"] is not None], 0.95)), 1),
            "처리량(건/s)": round(len(msgs) / dt), "모드": "저하모드" if fail_safe else "일반"}


base = make_messages(600, 0.21, seed=7)
surge = base + [m for k in range(5) for m in make_messages(600, 0.21, seed=20 + k, prefix=f"S{k}")]   # 평시 + 500%
CASES = [("평시", base, {}), ("트래픽 500% 증가", surge, {}), ("대역폭 90% 급감(200~500s)", base, dict(bw_drop=[(200, 500, 0.1)])),
         ("트래픽 500% + 대역폭 90% 급감", surge, dict(bw_drop=[(200, 500, 0.1)])),
         ("트래픽 500% + 재밍 120초", surge, dict(jam=[(200, 320)])), ("링크 두절 300초(대역폭 1%)", base, dict(bw_drop=[(200, 500, 0.01)]))]

rows = []
for i, (name, msgs, lk) in enumerate(CASES):
    link = Link(4000, seed=1, **lk)
    for fs in (False, True):
        rows.append({"상황": name, **run_case(msgs, link, fs, seed=i)})
df = pd.DataFrame(rows)
print(df.drop(columns=["처리량(건/s)"]).to_string(index=False))
print(f"\n시스템 다운 {df['다운'].sum()}건, 유실 {df['유실'].sum()}건(전달+보관=전체), 최소 처리량 {df['처리량(건/s)'].min():,}건/s")
hard = df[df["상황"] == "트래픽 500% + 대역폭 90% 급감"]
print("링크 두절(대역폭 1%) 구간에는 어떤 방식으로도 기한을 지킬 수 없다 -> 통신관에게 대체 수단(중계·전령 등) 검토를 요청하는 것이 안전 실패 동작이다.")
print(f"최악 상황 긴급 전문 기한 내 전달률: 일반 {hard.iloc[0]['긴급 기한내']:.1%} -> 성능 저하 모드 {hard.iloc[1]['긴급 기한내']:.1%}")
need = sum(m["size"] + 14 for m in surge if RANK[m["prec"]] <= 1 and 200 <= m["t0"] < 500) / 300
print(f"원인: 급감 구간 긴급 전문만 초당 {need:.0f}B가 필요한데 링크는 초당 {NOMINAL_BPS * 0.1 / 8:.0f}B -> 스케줄링으로 해결 불가, 전문 간소화·발송 통제를 지휘계통에 건의해야 한다.")

# 메모리 누수 점검: 같은 시나리오를 3회 반복해도 메모리가 계속 늘지 않는가
tracemalloc.start()
mem = []
peak = 0
for k in range(3):
    simulate(base, Link(4000, seed=1), **ADV, until=2000)
    mem.append(tracemalloc.get_traced_memory()[0] / 1e6); peak = max(peak, tracemalloc.get_traced_memory()[1] / 1e6)
tracemalloc.stop()
print(f"평시 시나리오 실행 중 최대 메모리 {peak:.1f} MB")
print("반복 실행 후 잔류 메모리(MB):", [round(m, 2) for m in mem], "-> 누수 의심" if mem[-1] > mem[0] * 1.5 + 1 else "-> 누수 징후 없음")

# 병목 함수 프로파일링
pr = cProfile.Profile(); pr.enable()
simulate(surge, Link(4000, seed=1, bw_drop=[(200, 500, 0.1)]), **ADV, until=2000)
pr.disable()
s = io.StringIO(); pstats.Stats(pr, stream=s).sort_stats("tottime").print_stats(4)
top = [ln.split() for ln in s.getvalue().splitlines() if ln.strip()[:1].isdigit() and "(" in ln and "function calls" not in ln][:3]
print("[병목 상위 3 함수: 호출 수 / 자체 소요(s) / 함수]")
for f in top:
    print(f"  {f[0]:>9} / {f[1]} / {' '.join(f[5:]).split('/')[-1][-45:]}")
print("개선 방향: 매 결정마다 전체 대기 전문을 훑는 부분을 긴급도별 힙(heapq)과 재전송 타이머 힙으로 바꾸면 대기 전문 수에 덜 민감해진다.")

plt.figure(figsize=(9, 4))
x = np.arange(len(CASES))
plt.bar(x - 0.2, df[df["모드"] == "일반"]["긴급 기한내"], 0.4, label="normal")
plt.bar(x + 0.2, df[df["모드"] == "저하모드"]["긴급 기한내"], 0.4, label="degraded mode (urgent only)")
plt.xticks(x, [f"case{i + 1}" for i in x]); plt.ylabel("urgent on-time ratio"); plt.ylim(0, 1.05); plt.legend()
plt.title("Stress test: urgent message delivery (synthetic)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
                   상황  전문  전달  보관  유실  다운  긴급 기한내  FLASH 95%지연(s)   모드
                   평시 130 130   0   0   0   1.000             2.0   일반
                   평시 130 130   0   0   0   1.000             1.7 저하모드
          트래픽 500% 증가 758 567 191   0   0   0.991             2.0   일반
          트래픽 500% 증가 758 579 179   0   0   0.991             2.2 저하모드
 대역폭 90% 급감(200~500s) 130 130   0   0   0   0.500            23.9   일반
 대역폭 90% 급감(200~500s) 130 130   0   0   0   0.667            18.7 저하모드
트래픽 500% + 대역폭 90% 급감 758 519 239   0   0   0.389            93.4   일반
트래픽 500% + 대역폭 90% 급감 758 528 230   0   0   0.398            97.8 저하모드
   트래픽 500% + 재밍 120초 758 547 211   0   0   0.770            14.8   일반
   트래픽 500% + 재밍 120초 758 563 195   0   0   0.858             6.7 저하모드
   링크 두절 300초(대역폭 1%) 130 130   0   0   0   0.389           259.8   일반
   링크 두절 300초(대역폭 1%) 130 130   0   0   0   0.500           175.0 저하모드

시스템 다운 0건, 유실 0건(전달+보관=전체), 최소 처리량 207건/s
링크 두절(대역폭 1%) 구간에는 어떤 방식으로도 기한을 지킬 수 없다 -> 통신관에게 대체 수단(중계·전령 등) 검토를 요청하는 것이 안전 실패 동작이다.
최악 상황 긴급 전문 기한 내 전달률: 일반 38.9% -> 성능 저하 모드 39.8%
원인: 급감 구간 긴급 전문만 초당 91B가 필요한데 링크는 초당 30B -> 스케줄링으로 해결 불가, 전문 간소화·발송 통제를 지휘계통에 건의해야 한다.
평시 시나리오 실행 중 최대 메모리 0.4 MB
반복 실행 후 잔류 메모리(MB): [0.01, 0.01, 0.01] -> 누수 징후 없음
[병목 상위 3 함수: 호출 수 / 자체 소요(s) / 함수]
     807785 / 0.476 / m08.py:80(policy_key)
          1 / 0.430 / m08.py:89(simulate)
       2132 / 0.408 / m08.py:110(<listcomp>)
개선 방향: 매 결정마다 전체 대기 전문을 훑는 부분을 긴급도별 힙(heapq)과 재전송 타이머 힙으로 바꾸면 대기 전문 수에 덜 민감해진다.
```
</details>
